# Is the cosine family the ceiling?

**Two experiments that test whether the gap between attribute-space scoring and any
cosine-space combiner is structural rather than a training deficiency.**

`docs/method.md` §7 reports 0.465 R@10 for attribute-space scoring against 0.267 for
CPAS-MLP, the best cosine-space method. The claim we want to defend in the report is
that **no combiner Φ emitting a single query vector can close that gap** — that the
limitation is the functional form, not the optimizer.

### The algebra the experiments come from

Expanding the expected Hamming term of the §5 score:

$$\mathbb{E}[\text{Hamming}] = \sum_a p_a(1-r_a) + (1-p_a)r_a = \sum_a p_a(1-2r_a) + \sum_a r_a$$

The second sum does not depend on the candidate, so with $s_a = 2r_a - 1 \in \{-1,+1\}$:

$$\text{rank by } -\mathbb{E}[\text{Hamming}] \;\equiv\; \text{rank by } \langle s, p(d)\rangle$$

**The main method is also an inner product** — just taken in $\mathbb{R}^{40}$, where the
criterion's coordinates are the axes, with a $\pm1$ query vector. A cosine method takes
its inner product in $\mathbb{R}^{512}$, where CLIP's pretraining fixed the geometry.

If the probe were linear (no sigmoid), $\langle s, Wd+b\rangle = \langle W^\top s, d\rangle
+ \text{const}$ and the two would coincide exactly, with $q = W^\top s$. So the entire
structural gap lives in what sits between: **the sigmoid** (which caps each attribute's
contribution, making the score non-compensatory — a linear functional is compensatory,
so a surplus on one attribute pays for a violation on another), **the constraint
indicator** (not a linear functional at all), and, for the MLP head, the fact that
$p(d)$ is genuinely nonlinear so no fixed $q$ exists.

### What each experiment measures

| | question | how |
|---|---|---|
| **A — oracle-centroid query** | is the ground-truth set even a *cosine-compact* region? | build `q` as the centroid of the answer set, i.e. with full knowledge of the answer, and rank by cosine |
| **B — cosine twin** | what do the sigmoid and the constraint term actually buy? | `q = normalize(Wᵀs)`: the main method's own score with the nonlinearities stripped out |

**Honest scope of A.** This is *not* a proven upper bound over all `q`: picking
`q = f_g` for any single ground-truth image `g` trivially gives R@1 = 1. It is the
ceiling for a query aimed at the answer *region*, which is the only thing a combiner can
do — Φ sees the reference and the constraints, never which individual database image is
the intended answer. If the centroid of the answer set cannot retrieve the answer set,
no combiner approximating that centroid can either.

### To run this you need

Neither experiment opens an image and neither runs CLIP. Everything is tensor algebra
over cached artifacts:

| file | size | needed by |
|---|---|---|
| `features/clip-vit-base-patch32_test.pt` | ~41 MB | everything |
| `celeba_evaluation.json` | small | everything |
| `results/probe_weights.pt` | ~82 KB | B, and the attribute-space reference row |
| test-split labels | ~800 KB | the §1 integrity check, V@10 |

Labels are read from `features/test_labels.pt` if present (dump it on a machine that
has the dataset with `torch.save(load_dataset(get_paths()).attr.bool(), ...)`), and
otherwise parsed straight from CelebA's metadata `.txt` files — **the image folder is
never touched**.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir():           # tolerate being run from a subfolder
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
torch.set_grad_enabled(False)

from src.attribute_retrieval import attribute_scores, expected_hamming, target_code
from src.criterion import MAX_HAMMING, satisfies
from src.evaluation import KS, _query_row, _with_mean_row, negation_subset
from src.probes import load_probes, load_raw_probes
from src.retrieval import parse_query

FEATURES_DIR = REPO_ROOT / "features"
ANNOTATIONS_PATH = REPO_ROOT / "celeba_evaluation.json"

SOURCE_CHUNK = 512      # references scored at once; lower it if RAM is tight
LAM_CONSTRAINT = 4.0    # validation-selected, docs/method.md S5
GAMMA = 0.6             # the fixed-rule reference row, docs/method-history.md S3

## 0. Loading — features, labels and probes, without the images

In [ ]:
def load_test_features() -> torch.Tensor:
    """(N, 512) L2-normalized CLIP features for the test split."""
    path = FEATURES_DIR / "clip-vit-base-patch32_test.pt"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Copy it from a machine that has run the pipeline "
            "(~41 MB), or regenerate it with scripts/run_baseline.py."
        )
    saved = torch.load(path, map_location="cpu", weights_only=True)
    feats = saved["features"] if isinstance(saved, dict) else saved
    return feats.float()


def load_test_labels(attributes: list[str]) -> torch.Tensor:
    """(N, 40) bool attribute labels for the test split, in dataset order.

    Prefers a dumped tensor; falls back to CelebA's metadata text files. Both
    paths avoid `src.data.load_dataset`, which requires the image folder to
    exist even though nothing here decodes an image.
    """
    dump = FEATURES_DIR / "test_labels.pt"
    if dump.is_file():
        return torch.load(dump, map_location="cpu", weights_only=True).bool()

    base = REPO_ROOT / "celeba"
    attr_txt, part_txt = base / "list_attr_celeba.txt", base / "list_eval_partition.txt"
    if not (attr_txt.is_file() and part_txt.is_file()):
        raise FileNotFoundError(
            f"Need either {dump} (~800 KB) or CelebA's metadata text files under "
            f"{base}. Neither requires the images."
        )
    # Test split is partition 2; row order is the file order, which is what the
    # torchvision CelebA class uses to build the indices the ground truth keys on.
    test_files = {
        line.split()[0]
        for line in part_txt.read_text().splitlines()
        if line.strip() and line.split()[1] == "2"
    }
    lines = attr_txt.read_text().splitlines()
    names = lines[1].split()
    rows = [ln.split() for ln in lines[2:] if ln.strip()]
    kept = [r for r in rows if r[0] in test_files]
    matrix = torch.tensor(
        [[v == "1" for v in r[1:]] for r in kept], dtype=torch.bool
    )
    # Reorder columns to the probes' attribute order rather than assuming it matches.
    return matrix[:, [names.index(a) for a in attributes]]

In [ ]:
features = load_test_features()
directions, _, attributes = load_probes(REPO_ROOT)   # L2-normalized, for composition
W, B, _ = load_raw_probes(REPO_ROOT)                 # raw weights, for probabilities
labels = load_test_labels(attributes)
annotations = json.loads(ANNOTATIONS_PATH.read_text())

attr_index = {name: i for i, name in enumerate(attributes)}
N_IMAGES, N_ATTR = labels.shape


def rows_for(names: list[str]) -> list[int]:
    """Attribute rows for a parsed query, tolerating 'Heavy Makeup' for 'Heavy_Makeup'."""
    out = []
    for name in names:
        key = name.replace(" ", "_")
        if key not in attr_index:
            raise KeyError(f"unknown attribute {name!r} (normalized to {key!r})")
        out.append(attr_index[key])
    return out


queries = [parse_query(entry["query"]) for entry in annotations]
query_rows = [(rows_for(pos), rows_for(neg)) for pos, neg in queries]
sources = [[int(k) for k in entry["ground_truth"]] for entry in annotations]

# Predicted codes from the linear probe. sigmoid(w.d + b) needs the RAW weights:
# load_probes normalizes its weights while the saved biases belong to the
# unnormalized ones, so pairing those two gives meaningless probabilities.
probs = torch.sigmoid(features @ W.T + B)
code = probs > 0.5

print(f"features {tuple(features.shape)}  labels {tuple(labels.shape)}")
print(f"{len(annotations)} queries, {sum(len(s) for s in sources)} (query, reference) pairs")
assert features.shape[0] == labels.shape[0], "features and labels are misaligned"

## 1. Integrity check — reconstruct the ground truth from the labels

`docs/method.md` §1 states that §3.1.1 was verified by exact set reconstruction on all
33,052 (query, reference) pairs. Re-running it here is a checksum on the whole data
path at once: if the reconstruction matches, the features are aligned to the right
indices, the labels are the right split, and we have not fallen into the
index-vs-filename trap the assignment spends half a page on (§3.1.2).

**If this cell reports any mismatch, stop — every number below is meaningless.**

In [ ]:
label_f = labels.float()
mismatches, checked = 0, 0

for entry, (pos_rows, neg_rows), srcs in zip(annotations, query_rows, sources):
    others = [a for a in range(N_ATTR) if a not in set(pos_rows) | set(neg_rows)]
    ok = satisfies(labels, pos_rows, neg_rows)                       # condition (1)
    for start in range(0, len(srcs), SOURCE_CHUNK):
        chunk = srcs[start : start + SOURCE_CHUNK]
        # Hard 0/1 inputs make expected_hamming the exact Hamming distance.
        hamming = expected_hamming(label_f, labels[chunk], others)   # (N, R)
        valid = ok.unsqueeze(1) & (hamming <= MAX_HAMMING)           # condition (2)
        for col, src in enumerate(chunk):
            expected = torch.zeros(N_IMAGES, dtype=torch.bool)
            expected[entry["ground_truth"][str(src)]] = True
            rebuilt = valid[:, col].clone()
            rebuilt[src] = False                    # the reference is never its own target
            mismatches += not torch.equal(rebuilt, expected)
            checked += 1

print(f"reconstructed {checked} (query, reference) pairs, {mismatches} mismatches")
print("OK - the S3.1.1 rule and the data path both check out" if mismatches == 0
      else "MISMATCH - do not trust anything below this cell")

## 2. Shared harness

Every method below differs only in how it scores `(reference, query) → (N,)`. Metrics
come from `src/evaluation.py` so they are computed exactly as in `results/`, and the
reference rows are recomputed inside this run — absolute numbers shift with a probe
refit, so only within-run deltas are comparable (`docs/method.md` §8).

We take `topk(10)` rather than a full `argsort`: `_query_row` only reads the first
`max(KS) = 10` columns, and the full sort would be a 19,962-wide int64 matrix per
reference block.

In [ ]:
def top_k_order(scores: torch.Tensor, srcs: list[int], k: int = max(KS)) -> torch.Tensor:
    """(R, k) database indices, best first, with each reference's own row excluded."""
    scores = scores.clone()
    rows = torch.arange(scores.shape[0])
    scores[rows, torch.tensor(srcs)] = float("-inf")
    return scores.topk(k, dim=1).indices


def benchmark(score_fn, name: str) -> pd.DataFrame:
    """Run `score_fn(entry, chunk, pos_rows, neg_rows) -> (R, N)` over the benchmark."""
    rows = []
    for entry, (pos_rows, neg_rows), srcs in zip(annotations, query_rows, sources):
        order = torch.cat([
            top_k_order(
                score_fn(entry, srcs[i : i + SOURCE_CHUNK], pos_rows, neg_rows),
                srcs[i : i + SOURCE_CHUNK],
            )
            for i in range(0, len(srcs), SOURCE_CHUNK)
        ])
        rows.append(_query_row(entry, order, srcs, labels, pos_rows, neg_rows))
    print(f"{name:38s} done")
    return _with_mean_row(rows).assign(method=name)

## 3. Reference rows

Two anchors, so the experiments are read against something measured in the same run:
the fixed-rule probe composition (the cosine-space baseline, ~0.210) and attribute-space
scoring (the current method, ~0.465).

In [ ]:
def score_fixed_rule(entry, chunk, pos_rows, neg_rows):
    """q = normalize(gamma * v_ref + sum(pos_dirs) - sum(neg_dirs)), ranked by cosine."""
    edit = directions[pos_rows].sum(0) - directions[neg_rows].sum(0)
    q = GAMMA * features[chunk] + edit
    return (q / q.norm(dim=1, keepdim=True)) @ features.T


def score_attribute_space(entry, chunk, pos_rows, neg_rows):
    """The docs/method.md S5 score, cosine term off (w_cos = 0)."""
    targets = target_code(code[chunk], pos_rows, neg_rows)
    return attribute_scores(probs, code, targets, pos_rows, neg_rows,
                            lam_constraint=LAM_CONSTRAINT)


results = [
    benchmark(score_fixed_rule, "probe composition (gamma=0.6)"),
    benchmark(score_attribute_space, "attribute space (linear probe)"),
]

## 4. Experiment A — the oracle-centroid query

`q` is the normalized centroid of the ground-truth set for that reference: the query
vector built *knowing the answer*. It maximizes total cosine similarity to the answer
set over all unit vectors, so it is the best a set-aimed query can do.

Read the result as: **if this is not far above CPAS-MLP's 0.267, then no combiner in
cosine space has much room left**, because a trained Φ is at best approximating this
vector from the reference and the constraints alone.

In [ ]:
def score_oracle_centroid(entry, chunk, pos_rows, neg_rows):
    q = torch.stack([
        features[entry["ground_truth"][str(src)]].mean(0) for src in chunk
    ])
    return (q / q.norm(dim=1, keepdim=True)) @ features.T


results.append(benchmark(score_oracle_centroid, "ORACLE centroid query (A)"))

## 5. Experiment B — the cosine twin

`q = normalize(Wᵀs)` with `s` the signed target code. Two variants:

- **B1, Hamming only** — `s` restricted to the ~38 non-queried attributes. This is the
  exact linear twin of the main method's first term: identical, except the sigmoid is
  gone.
- **B2, all 40 bits** — `s` over every attribute. Since `target_code` has already forced
  the queried bits to ±1, this folds the constraints in as the linear analogue of the
  `λ` indicator, with no extra knob to tune.

The gap between these rows and the attribute-space row is **exactly what the
nonlinearities buy**, measured rather than argued.

Two things worth noting about `q = Wᵀs`, both departures from `compose_probe`: it uses
all 38 non-queried bits rather than only the queried ones, and it weights each attribute
by `‖w_a‖` — steeper probes count more, which is the per-attribute Hamming weighting of
`docs/method.md` §9.5 arriving for free. It also contains **no `v_ref` term at all**:
identity enters as the reference's *code*, not as a direction to stay near.

In [ ]:
def signed_target(chunk, pos_rows, neg_rows) -> torch.Tensor:
    """(R, A) in {-1, +1}: the target code with s_a = 2*r_a - 1."""
    return 2.0 * target_code(code[chunk], pos_rows, neg_rows).float() - 1.0


def score_twin_hamming(entry, chunk, pos_rows, neg_rows):
    others = [a for a in range(N_ATTR) if a not in set(pos_rows) | set(neg_rows)]
    q = signed_target(chunk, pos_rows, neg_rows)[:, others] @ W[others]
    return (q / q.norm(dim=1, keepdim=True)) @ features.T


def score_twin_full(entry, chunk, pos_rows, neg_rows):
    q = signed_target(chunk, pos_rows, neg_rows) @ W
    return (q / q.norm(dim=1, keepdim=True)) @ features.T


results.append(benchmark(score_twin_hamming, "cosine twin, Hamming only (B1)"))
results.append(benchmark(score_twin_full, "cosine twin, all 40 bits (B2)"))

## 6. Results

In [ ]:
OUT_DIR = REPO_ROOT / "results"
OUT_DIR.mkdir(exist_ok=True)

per_query = pd.concat(results, ignore_index=True)
metric_cols = [f"R@{k}" for k in KS] + [f"P@{k}" for k in KS] + ["V@10"]

# Keyed by method rather than positional, so reordering `results` cannot
# silently attach the wrong negation subset to a row.
neg_r10 = {df["method"].iloc[0]: negation_subset(df) for df in results}
summary = (
    per_query[per_query["query"] == "MEAN"]
    .set_index("method")[metric_cols]
)
summary["neg_R@10"] = [neg_r10[m] for m in summary.index]

summary.to_csv(OUT_DIR / "cosine_ceiling.csv")
per_query.to_csv(OUT_DIR / "cosine_ceiling_per_query.csv", index=False)
summary.style.format("{:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
order = summary["R@10"].sort_values()
colors = ["tab:gray" if "ORACLE" not in m else "tab:orange" for m in order.index]
ax.barh(range(len(order)), order.values, color=colors)
ax.set_yticks(range(len(order)), order.index, fontsize=9)
ax.axvline(0.267, ls="--", lw=1, color="tab:red")
ax.text(0.267, -0.9, " CPAS-MLP (docs/method.md S7)", color="tab:red", fontsize=8, va="top")
ax.set_xlabel("R@10, mean over the benchmark queries")
ax.set_title("Cosine-space methods against attribute-space scoring")
for i, v in enumerate(order.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "cosine_ceiling.png", dpi=150)
plt.show()

## 7. Diagnostic — where the correct answers actually sit

`docs/method.md` §1 claims the correct answers sit at cosine ranks in the thousands. This
measures it directly: the rank of the *best* ground-truth image, under the oracle-centroid
query and under the fixed rule.

If even the oracle-centroid query leaves the nearest correct answer at rank ~100, the
ground-truth set is not a cap of the sphere and cosine ranking cannot reach it — which is
the geometric statement the whole argument rests on.

In [ ]:
def best_gt_rank(score_fn) -> torch.Tensor:
    """Rank (1-based) of the highest-scoring ground-truth image, per (query, reference)."""
    out = []
    for entry, (pos_rows, neg_rows), srcs in zip(annotations, query_rows, sources):
        for start in range(0, len(srcs), SOURCE_CHUNK):
            chunk = srcs[start : start + SOURCE_CHUNK]
            scores = score_fn(entry, chunk, pos_rows, neg_rows)
            gt = torch.zeros_like(scores, dtype=torch.bool)
            for col, src in enumerate(chunk):
                gt[col, entry["ground_truth"][str(src)]] = True
            best = scores.masked_fill(~gt, float("-inf")).max(dim=1).values
            out.append((scores > best.unsqueeze(1)).sum(dim=1) + 1)
    return torch.cat(out)


ranks = {
    "oracle centroid (A)": best_gt_rank(score_oracle_centroid),
    "probe composition": best_gt_rank(score_fixed_rule),
    "attribute space": best_gt_rank(score_attribute_space),
}
pd.DataFrame(
    {name: {"median": float(r.float().median()),
            "90th pct": float(r.float().quantile(0.9)),
            "frac in top 10": float((r <= 10).float().mean())}
     for name, r in ranks.items()}
).T.style.format({"median": "{:.0f}", "90th pct": "{:.0f}", "frac in top 10": "{:.3f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
bins = torch.logspace(0, torch.log10(torch.tensor(float(N_IMAGES))), 40).numpy()
for name, r in ranks.items():
    ax.hist(r.numpy(), bins=bins, histtype="step", lw=1.8, label=name)
ax.axvline(10, ls="--", lw=1, color="k")
ax.text(11, ax.get_ylim()[1] * 0.9, "top 10", fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("rank of the best ground-truth image")
ax.set_ylabel("(query, reference) pairs")
ax.legend(fontsize=8)
ax.set_title("How far the nearest correct answer sits under each ranking")
fig.tight_layout()
fig.savefig(OUT_DIR / "cosine_ceiling_ranks.png", dpi=150)
plt.show()

## 8. How to read this

Fill these in once the cells have run — the claims are only worth making with the
numbers attached.

- **If A lands well below the attribute-space row**, the report can state that the
  ground-truth set is not cosine-compact and that *any* Φ emitting one vector is capped
  there, with the caveat of §0 stated explicitly. This is what closes the "maybe CPAS was
  just trained badly" objection, and it pairs with the corrected miner: the module was
  trained against the right criterion *and* its family has a ceiling.
- **If A lands near or above the attribute-space row**, the structural claim is wrong as
  stated and should be dropped. The gap would then be an optimization gap, and CPAS
  would be worth far more investment than §5 of `docs/method.md` currently assumes.
- **B1 vs the attribute-space row** isolates the sigmoid and the constraint term. A large
  gap is the quantitative version of "a linear functional is compensatory".
- **B2 vs B1** is what the queried bits contribute once folded in linearly.

Remember the project's own resolution limit: **differences below 0.02 R@10 are not
resolved** (`docs/method.md` §8), and none of these rows involve training, so seed noise
is not a factor here — but the comparison against the published 0.267 for CPAS-MLP is
across runs, and only the rows computed in this notebook are strictly within-run.